# SumExplicitStack (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/SumExplicitStack_python.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `SumExplicitStack`: the algorithm in Python, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/SumExplicitStack_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/SumExplicitStack_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fib_rec` (snake_case).

## From recursion to an explicit stack: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the sum $a[1]+\dots+a[n]$. The iterative version of the halving sum: the stack $P$ contains the pairs $(l,r)$ of the sub-arrays still to be summed and does the job of the call stack. The stack never contains more than $\Theta(\log_2 n)$ pairs.

On the array $a=[4,7,1,9]$ of the chapter we print the trace (with indices from 0: the pair $(1,4)$ of the chapter is here $(0,3)$; in each row the top of the stack is on the right) and we compare with `SumIterative`, `sumRecHalving` and `sumRecDecrease` of the code of the course, with the maximum depth of the recursion. Then we print the call stack of `sumRecDecrease(a, 3)` on $a=[4,7,1]$ (example of the chapter): the additions are performed only *after* the base case, while the stack is emptied, and at some point the active calls are $n+1$.

In [1]:
max_p = 0   # maximum size of the stack P

def sum_explicit_stack(a, n):
    """SumExplicitStack(a, n): sum of a[0..n-1] with an explicit stack."""
    global max_p
    s = 0
    P = []                          # stack of the sub-arrays to be summed
    P.append((0, n - 1))
    max_p = max(max_p, len(P))
    while len(P) > 0:
        l, r = P.pop()
        if l == r:
            s = s + a[l]            # base case
        else:
            mid = l + (r - l) // 2
            P.append((mid + 1, r))
            P.append((l, mid))
            max_p = max(max_p, len(P))
    return s

def fmt_p(P):
    return "[" + ",".join(f"({x},{y})" for x, y in P) + "]"

def sum_explicit_stack_trace(a, n):
    s = 0
    P = [(0, n - 1)]
    while len(P) > 0:
        l, r = P.pop()
        if l == r:
            s = s + a[l]
            print(f"pop ({l},{r}): base case, s={s} => P={fmt_p(P)}")
        else:
            mid = l + (r - l) // 2
            P.append((mid + 1, r))
            P.append((l, mid))
            print(f"pop ({l},{r}): mid={mid}, push ({mid + 1},{r}) and ({l},{mid}) => P={fmt_p(P)}, s={s}")
    return s

# ---- the other sums of the course code, with the depth ----
depth = 0
max_depth = 0

def sum_it(a, n):
    s = 0
    for i in range(n):
        s = s + a[i]
    return s

def sum_rec_halving(a, l, r):
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)
    if l == r:
        result = a[l]
    else:
        mid = l + (r - l) // 2
        result = sum_rec_halving(a, l, mid) + sum_rec_halving(a, mid + 1, r)
    depth -= 1
    return result

def sum_rec_decrease(a, n):
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)
    if n == 0:
        result = 0
    else:
        result = sum_rec_decrease(a, n - 1) + a[n - 1]
    depth -= 1
    return result

a = [4, 7, 1, 9]
n = len(a)
print("a =", a, "  n =", n)
print("SumExplicitStack(a, 4):")
res = sum_explicit_stack_trace(a, n)
max_p = 0
sum_explicit_stack(a, n)
print(f"result {res}, maximum size of P = {max_p}")
print()
print("sumIt                =", sum_it(a, n))
max_depth = 0
print("sumRecHalving        =", sum_rec_halving(a, 0, n - 1), "  maximum depth =", max_depth)
max_depth = 0
print("sumRecDecrease       =", sum_rec_decrease(a, n), "  maximum depth =", max_depth)

# ---- the call stack of sumRecDecrease(a, 3) ----
step = 0
max_active = 0

def emit(stack, action):
    global step, max_active
    step += 1
    max_active = max(max_active, len(stack))
    print(f"{step:>2d} | {' '.join(stack)} | {action}")

def sum_rec_decrease_trace(a, n, stack):
    stack.append(f"s({n})")
    if n == 0:
        emit(stack, "n=0: return 0")
        stack.pop()
        return 0
    emit(stack, f"n={n} != 0: call s({n - 1}), then add a[{n - 1}]")
    r = sum_rec_decrease_trace(a, n - 1, stack)
    emit(stack, f"return {r}+a[{n - 1}]={r + a[n - 1]}")
    stack.pop()
    return r + a[n - 1]

b = [4, 7, 1]
print()
print("a =", b, "  sumRecDecrease(a, 3)")
print(" # | stack of the active calls | action")
res = sum_rec_decrease_trace(b, 3, [])
print(f"result {res}, active calls at most = {max_active} = n+1")

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) and (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) and (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): base case, s=4 => P=[(2,3),(1,1)]
pop (1,1): base case, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) and (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): base case, s=12 => P=[(3,3)]
pop (3,3): base case, s=21 => P=[]
result 21, maximum size of P = 3

sumIt                = 21
sumRecHalving        = 21   maximum depth = 3
sumRecDecrease       = 21   maximum depth = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | stack of the active calls | action
 1 | s(3) | n=3 != 0: call s(2), then add a[2]
 2 | s(3) s(2) | n=2 != 0: call s(1), then add a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: call s(0), then add a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
result 12, active calls at most = 4 = n+1


**Try it yourself.** Change the array: the explicit stack never contains more than $\lceil\log_2 n\rceil+1$ pairs.

In [2]:
# Change the array: the explicit stack never contains more than ceil(log2 n) + 1 pairs.
c = list(range(1, 101))
max_p = 0
print(sum_explicit_stack(c, len(c)), max_p)

5050 8


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).